In [2]:
# from 01-data.ipynb
import requests

URL_PREFIX='https://datatalks.club/faq'

path = '/json/courses.json'
response = requests.get(url=f'{URL_PREFIX}{path}')
courses_raw = response.json()

documents = []
for course in courses_raw:
    course_path = f'{URL_PREFIX}{course['path']}'
    course_response = requests.get(url=course_path)
    course_response.raise_for_status()
    course_faqs = course_response.json()

    documents.extend(course_faqs)

print(f'Total number of faq documents: {len(documents)}')

Total number of faq documents: 1346


In [3]:
from minsearch import Index

# we'll index question, section, answer fields as text (they'll be tokenised and ranked), and course field as the keyword (for filtering)
index = Index(
    text_fields=['question','section','answer'],
    keyword_fields=['course']
)
# perform indexing
index.fit(documents)

In [ ]:
# function to perform search for a user query in the indexed data
def search(question, course='llm-zoomcamp'):
    # give 'question' more weightage (more than default weightage of 1.0) and 'section', a less weightage; this means the query words appearing in question field is a stronger signal than
    # them appearing in the section name
    boost_dict = {'question':2.0, 'section':0.5}
    # return the results only from the 'llm-zoomcamp' course; without this, we will get results from all the courses
    filter_dict={'course':course}

    return index.search(
        query=question,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=5
    )

question = 'I just discovered the course. Can I join now?'
search_results = search(question) # these search results will be sent as a context to the llm
display(search_results)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c